# Huấn luyện M2/M3

M1 đã hoàn tất. Hai nhánh khởi đầu từ cùng checkpoint-3683: M2 học chosen, M3 học chosen/rejected.

Bộ cuối đã khóa 186 cặp. Mặc định chỉ kiểm đầu vào, không train. Thứ tự: setup → preflight → kiểm GPU → train → đọc kết quả.

[M1: code và output đã chạy](https://colab.research.google.com/drive/13Abjl7kJyDKzbxyYsCetJ540OzdK2P_E). Bản local: notebooks/archive/03_m1_completed_with_outputs_20261013.ipynb.


In [ ]:
import hashlib
import json
import subprocess
import sys
from pathlib import Path

from google.colab import drive

if not Path('/content/drive/MyDrive').is_dir():
    drive.mount('/content/drive', timeout_ms=600000)
ROOT = Path('/content/drive/MyDrive/Vietnamese_VQA_NLE')
if not ROOT.is_dir():
    raise FileNotFoundError('Project folder is missing from this Drive')
LAUNCH_PATH = ROOT / 'source/preference_training_launch_v1.json'
if not LAUNCH_PATH.is_file():
    raise FileNotFoundError(f'Launch record is missing: {LAUNCH_PATH}')
print('Project storage ready:', ROOT)


In [ ]:
launch = json.loads(LAUNCH_PATH.read_text(encoding='utf-8'))
bootstrap = ROOT / 'source/bootstrap_cloud_source.py'
bootstrap_sha = '391172ed1fbf39d0eeeef2c9aa3d056d7770085d0a10574d571470d7f2bbef40'
if hashlib.sha256(bootstrap.read_bytes()).hexdigest() != bootstrap_sha:
    raise ValueError('Bootstrap source changed')
release = launch['source_release']
if not release.startswith('source_') or any(char in release for char in '/\\:'):
    raise ValueError('Invalid source release name')
command = [sys.executable, str(bootstrap), '--archive',
           str(ROOT / 'source/releases' / f'{release}.zip'),
           '--sha256', launch['source_sha256'], '--work-root', '/content/vqa_sources']
REPO = Path(subprocess.check_output(command, text=True).strip())
sys.path.insert(0, str(REPO / 'src'))
from vqa_explanation.utils.process import run_command  # noqa: E402

ENV_DIR = Path('/content/vqa_envs') / release
PYTHON = ENV_DIR / 'bin/python'
print('Source:', release)
run_command([sys.executable, str(REPO / 'scripts/setup_preference_environment.py'),
             '--environment-dir', str(ENV_DIR)], cwd=REPO)
print('Training interpreter:', PYTHON)


In [ ]:
from vqa_explanation.training.preference_launch import launch_command, stage_output
from vqa_explanation.utils.environment import load_environment

MODE = 'inspect'
RUN_FOLDER = ROOT / launch['run_group']
RESUME = {'m2': None, 'm3': None}
ATTEMPT = {'m2': 'initial', 'm3': 'initial'}

if launch.get('data_locked') is True:
    environment = load_environment(REPO / launch['environment'])
    if not environment['image_root'].exists():
        bundle = ROOT / launch['dataset_bundle']
        digest = hashlib.sha256((bundle / 'bundle.json').read_bytes()).hexdigest()
        if digest != launch['dataset_bundle_sha256']:
            raise ValueError('Dataset bundle changed')
        run_command([str(PYTHON), str(REPO / 'scripts/verify_coco_bundle.py'),
                        str(bundle), '--extract-to', str(environment['image_root'])],
                    cwd=REPO)

for arm in ('m2', 'm3'):
    command = launch_command(REPO, LAUNCH_PATH, ROOT, arm, 'inspect', RUN_FOLDER,
                             python_executable=PYTHON)
    run_command(command, cwd=REPO)


In [ ]:
command = launch_command(REPO, LAUNCH_PATH, ROOT, 'm2', MODE, RUN_FOLDER,
                         RESUME['m2'], ATTEMPT['m2'], python_executable=PYTHON)
run_command(command, cwd=REPO)


In [ ]:
command = launch_command(REPO, LAUNCH_PATH, ROOT, 'm3', MODE, RUN_FOLDER,
                         RESUME['m3'], ATTEMPT['m3'], python_executable=PYTHON)
run_command(command, cwd=REPO)


In [ ]:
for arm in ('m2', 'm3'):
    result = stage_output(RUN_FOLDER, arm, MODE, ATTEMPT[arm]) / 'training_result.json'
    if result.is_file():
        report = json.loads(result.read_text(encoding='utf-8'))
        fields = ('status', 'global_step', 'epoch', 'training_loss', 'peak_vram_gib',
                  'policy_updated', 'reference_unchanged', 'saved_policy_verified')
        print(arm, json.dumps({key: report.get(key) for key in fields}, indent=2))
    else:
        print(arm, 'No training result:', result)
